# 联合SOXX桥接0bp账户：最小失败归因

只读封存候选与f696 A0原生Qlib Recorder、信息模型DatasetH；核对预测正值支持与944行净值／费用路径，不重训或重放账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_joint_soxx_bridge_0bp_20261024"
plan_path=card/"failure_attribution_plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
for file,key in [(card/"plan.json","account_plan_sha256"),(card/"preflight_result.json","account_preflight_sha256"),(card/"result.json","account_result_sha256")]:assert sha(file)==src[key]
result=json.loads((card/"result.json").read_text());pre=json.loads((card/"preflight_result.json").read_text())
assert result["new_native_accounts"]==1 and result["NAS_readback_verified"] and pre["trigger_days"]==0
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
account_art=card/"qlib_recorder"/src["account_experiment_id"]/src["account_recorder_id"]/"artifacts"
A0=root/src["A0_card"]
a0_art=A0/"qlib_recorder"/src["A0_experiment_id"]/src["A0_recorder_id"]/"artifacts"
info=root/src["information_card"]
info_result=json.loads((info/"result.json").read_text())
info_art=info/"qlib_recorder"/info_result["experiment_id"]/info_result["recorder_id"]/"artifacts"
assert sha(account_art/"predictions/joint_margin_full")==src["account_prediction_sha256"]
assert sha(account_art/"native/candidate_0")==src["account_native_sha256"]
assert sha(a0_art/"native/A_active_research_replay_0")==src["A0_native_sha256"]
assert sha(info_art/"inputs/dataset")==src["information_dataset_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2023-01-04",end_time="2026-10-09",freq="day"))
assert calendar.is_monotonic_increasing and len(calendar)>=943
with R.uri_context(str((card/"qlib_recorder").resolve())):
 rec=R.get_recorder(experiment_id=src["account_experiment_id"],recorder_id=src["account_recorder_id"])
 predictions=rec.load_object("predictions/joint_margin_full")
 book=rec.load_object("native/candidate_0")
with R.uri_context(str((A0/"qlib_recorder").resolve())):
 rec=R.get_recorder(experiment_id=src["A0_experiment_id"],recorder_id=src["A0_recorder_id"])
 original=rec.load_object("native/A_active_research_replay_0")
with R.uri_context(str((info/"qlib_recorder").resolve())):
 rec=R.get_recorder(experiment_id=info_result["experiment_id"],recorder_id=info_result["recorder_id"])
 dataset=rec.load_object("inputs/dataset")
assert len(predictions)==944 and pd.DatetimeIndex(predictions.index).equals(pd.DatetimeIndex(book["authority"].signal_date))
assert pd.DatetimeIndex(book["authority"].trade_date).isin(calendar).all()
assert np.isfinite(predictions.to_numpy(float)).all()
assert int((predictions>0).sum())==0
candidate=book["authority"].reset_index(drop=True)
source=original["authority"].reset_index(drop=True)
assert len(candidate)==len(source)==944
same_columns=list(candidate.columns)==list(source.columns)
common=sorted(set(candidate.columns)&set(source.columns))
numeric=[c for c in common if pd.api.types.is_numeric_dtype(candidate[c]) and pd.api.types.is_numeric_dtype(source[c])]
max_numeric=max(float(np.max(np.abs(candidate[c].to_numpy(float)-source[c].to_numpy(float)))) for c in numeric)
dates_equal=all(pd.to_datetime(candidate[c]).equals(pd.to_datetime(source[c])) for c in ["signal_date","trade_date","exit_date"])
assert same_columns and dates_equal and max_numeric<1e-12
train=dataset.prepare("train",col_set=["label","meta"])
held=dataset.prepare("held",col_set=["label","meta"])
train=train.loc[train[("meta","train_mature")].astype(bool)]
held=held.loc[held[("meta","held_mature")].astype(bool)]
assert len(train)==3014 and len(held)==940
joint_train=np.minimum(-train[("label","cash_advantage_H5")].to_numpy(float),-train[("label","SOXL_relative_H5")].to_numpy(float))
joint_held=np.minimum(-held[("label","cash_advantage_H5")].to_numpy(float),-held[("label","SOXL_relative_H5")].to_numpy(float))
assert np.isfinite(joint_train).all() and np.isfinite(joint_held).all()
train_dates=pd.DatetimeIndex(train.index.get_level_values("datetime"));held_dates=pd.DatetimeIndex(held.index.get_level_values("datetime"))
train_year={str(year):{"mature_rows":int((train_dates.year==year).sum()),"positive_joint_margin_rows":int(((train_dates.year==year)&(joint_train>0)).sum())} for year in sorted(set(train_dates.year))}
held_year={str(year):{"mature_rows":int((held_dates.year==year).sum()),"positive_joint_margin_rows":int(((held_dates.year==year)&(joint_held>0)).sum()),
                      "positive_model_prediction_rows":int(((pd.DatetimeIndex(predictions.index).year==year)&(predictions.to_numpy(float)>0)).sum()),
                      "prediction_max":float(predictions[pd.DatetimeIndex(predictions.index).year==year].max())} for year in (2023,2024,2025,2026)}
attribution={"status":"fixed_account_no_trigger_exact_A0_path_failure_attribution",
             "plan_sha256":sha(plan_path),"account_result_sha256":src["account_result_sha256"],
             "predictions": {"rows":len(predictions),"minimum":float(predictions.min()),"median":float(predictions.median()),"maximum":float(predictions.max()),"positive_count":int((predictions>0).sum())},
             "joint_target":{"train_mature":len(train),"train_positive":int((joint_train>0).sum()),"held_mature":len(held),"held_positive":int((joint_held>0).sum()),"train_by_year":train_year,"held_by_year":held_year},
             "native_authority_parity":{"same_columns":same_columns,"date_columns_equal":dates_equal,"numeric_columns_compared":len(numeric),"max_abs_numeric_difference":max_numeric},
             "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_Recorders":0,
             "scientific_decision":"not_promoted_fixed_joint_SOXX_bridge_0bp_no_action", "threshold_or_transfer_recalibrated":False,
             "next_action":"Close this exact prediction>0 and10%/10% transfer version. Information MSE gains without actionable positive predictions did not change f696. A materially different decision-time source/receiver is required; do not lower the score threshold on reused held history."}
(card/"failure_attribution.json").write_text(json.dumps(attribution,ensure_ascii=False,indent=2)+"\n")
print({"prediction_range":[attribution["predictions"][k] for k in ("minimum","median","maximum")],"train_positive":attribution["joint_target"]["train_positive"],"held_positive":attribution["joint_target"]["held_positive"],"native_max_error":max_numeric})


{'prediction_range': [-0.11708031676302125, -0.03885942399782488, -0.003223862498271328], 'train_positive': 79, 'held_positive': 33, 'native_max_error': 0.0}


## 中文结论

预测全负、联合标签稀有；候选与f696原生账户逐日完全相同。详见 `failure_attribution.json` 与主结论。